# Kalshi Trading Bot — Explorer
Use this notebook to interactively inspect markets, signals, positions, and place orders.

In [1]:
import os

# Make sure imports resolve from this folder
os.chdir(os.path.dirname(os.path.abspath('explore.ipynb')))

import pandas as pd
from client import KalshiClient
from markets import get_open_weather_markets
from strategy import get_signals

client = KalshiClient()
print('Connected.')

Kalshi client initialized (LIVE)
Connected.


## Account

In [2]:
balance_data = client.get_balance()
balance = balance_data['balance'] / 100
print(f'Balance: ${balance:.2f}')

Balance: $75.54


In [3]:
positions = client.get_positions()
if positions:
    pd.DataFrame(positions)
else:
    print('No open positions.')

No open positions.


In [4]:
orders = client.get_orders()
if orders:
    pd.DataFrame(orders)
else:
    print('No orders.')

## Weather Markets

In [5]:
markets = get_open_weather_markets(client)
print(f'{len(markets)} open weather markets')

df_markets = pd.DataFrame([{
    'ticker':    m.get('ticker'),
    'series':    m.get('series_ticker'),
    'title':     m.get('title', '').replace('**', ''),
    'strike':    m.get('floor_strike'),
    'yes_bid':   float(m.get('yes_bid_dollars') or 0),
    'yes_ask':   float(m.get('yes_ask_dollars') or 0),
    'volume':    float(m.get('volume_fp') or 0),
    'close_time': m.get('close_time'),
} for m in markets])

df_markets

50 open weather markets


,ticker,series,title,strike,yes_bid,yes_ask,volume,close_time
0,KXHIGHNY-26APR24-T67,KXHIGHNY,"Will the high temp in NYC be >67° on Apr 24, 2...",67.0,0.06,0.08,3124.23,2026-04-25T04:59:00Z
1,KXHIGHNY-26APR24-T60,KXHIGHNY,"Will the high temp in NYC be <60° on Apr 24, 2...",NaN,0.01,0.02,1353.54,2026-04-25T04:59:00Z
2,KXHIGHNY-26APR24-B66.5,KXHIGHNY,"Will the high temp in NYC be 66-67° on Apr 24,...",66.0,0.41,0.43,4409.76,2026-04-25T04:59:00Z
3,KXHIGHNY-26APR24-B64.5,KXHIGHNY,"Will the high temp in NYC be 64-65° on Apr 24,...",64.0,0.34,0.36,3939.06,2026-04-25T04:59:00Z
4,KXHIGHNY-26APR24-B62.5,KXHIGHNY,"Will the high temp in NYC be 62-63° on Apr 24,...",62.0,0.14,0.15,3361.01,2026-04-25T04:59:00Z
5,KXHIGHNY-26APR24-B60.5,KXHIGHNY,"Will the high temp in NYC be 60-61° on Apr 24,...",60.0,0.01,0.02,571.80,2026-04-25T04:59:00Z
6,KXHIGHNY-26APR23-T75,KXHIGHNY,"Will the high temp in NYC be >75° on Apr 23, 2...",75.0,0.00,0.01,29030.14,2026-04-24T04:59:00Z
7,KXHIGHNY-26APR23-T68,KXHIGHNY,"Will the high temp in NYC be <68° on Apr 23, 2...",NaN,0.00,0.01,7875.58,2026-04-24T04:59:00Z
8,KXHIGHNY-26APR23-B74.5,KXHIGHNY,"Will the high temp in NYC be 74-75° on Apr 23,...",74.0,0.06,0.07,60764.86,2026-04-24T04:59:00Z
9,KXHIGHNY-26APR23-B72.5,KXHIGHNY,"Will the high temp in NYC be 72-73° on Apr 23,...",72.0,0.93,0.96,57131.94,2026-04-24T04:59:00Z


In [6]:
# Filter to a specific series
df_markets[df_markets['series'] == 'KXHIGHNY']

,ticker,series,title,strike,yes_bid,yes_ask,volume,close_time
0,KXHIGHNY-26APR24-T67,KXHIGHNY,"Will the high temp in NYC be >67° on Apr 24, 2...",67.0,0.06,0.08,3124.23,2026-04-25T04:59:00Z
1,KXHIGHNY-26APR24-T60,KXHIGHNY,"Will the high temp in NYC be <60° on Apr 24, 2...",NaN,0.01,0.02,1353.54,2026-04-25T04:59:00Z
2,KXHIGHNY-26APR24-B66.5,KXHIGHNY,"Will the high temp in NYC be 66-67° on Apr 24,...",66.0,0.41,0.43,4409.76,2026-04-25T04:59:00Z
3,KXHIGHNY-26APR24-B64.5,KXHIGHNY,"Will the high temp in NYC be 64-65° on Apr 24,...",64.0,0.34,0.36,3939.06,2026-04-25T04:59:00Z
4,KXHIGHNY-26APR24-B62.5,KXHIGHNY,"Will the high temp in NYC be 62-63° on Apr 24,...",62.0,0.14,0.15,3361.01,2026-04-25T04:59:00Z
5,KXHIGHNY-26APR24-B60.5,KXHIGHNY,"Will the high temp in NYC be 60-61° on Apr 24,...",60.0,0.01,0.02,571.80,2026-04-25T04:59:00Z
6,KXHIGHNY-26APR23-T75,KXHIGHNY,"Will the high temp in NYC be >75° on Apr 23, 2...",75.0,0.00,0.01,29030.14,2026-04-24T04:59:00Z
7,KXHIGHNY-26APR23-T68,KXHIGHNY,"Will the high temp in NYC be <68° on Apr 23, 2...",NaN,0.00,0.01,7875.58,2026-04-24T04:59:00Z
8,KXHIGHNY-26APR23-B74.5,KXHIGHNY,"Will the high temp in NYC be 74-75° on Apr 23,...",74.0,0.06,0.07,60764.86,2026-04-24T04:59:00Z
9,KXHIGHNY-26APR23-B72.5,KXHIGHNY,"Will the high temp in NYC be 72-73° on Apr 23,...",72.0,0.93,0.96,57131.94,2026-04-24T04:59:00Z


## Signals

In [7]:
signals = get_signals(markets, balance)
if signals:
    pd.DataFrame(signals)[['ticker','series','city','side','strike','forecast_temp','forecast_prob','market_price','edge','contracts']]
else:
    print('No signals above threshold.')

  SIGNAL KXHIGHNY-26APR24-T67 | BUY YES @ 0.08 | forecast=34.5% market=8.0% edge=26.5% | 20 contracts
  SIGNAL KXHIGHNY-26APR24-T60 | BUY YES @ 0.02 | forecast=15.9% market=2.0% edge=13.9% | 20 contracts
  SIGNAL KXHIGHNY-26APR24-B66.5 | BUY NO @ 0.59 | forecast=14.6% market=59.0% edge=26.4% | 6 contracts
  SIGNAL KXHIGHNY-26APR24-B64.5 | BUY NO @ 0.66 | forecast=15.9% market=66.0% edge=18.1% | 5 contracts
  SIGNAL KXHIGHNY-26APR24-B60.5 | BUY YES @ 0.02 | forecast=11.6% market=2.0% edge=9.6% | 20 contracts
  SIGNAL KXHIGHMIA-26APR24-T85 | BUY YES @ 0.02 | forecast=21.2% market=2.0% edge=19.2% | 20 contracts
  SIGNAL KXHIGHMIA-26APR24-T78 | BUY YES @ 0.08 | forecast=27.4% market=8.0% edge=19.4% | 20 contracts
  SIGNAL KXHIGHMIA-26APR24-B82.5 | BUY NO @ 0.54 | forecast=14.6% market=54.0% edge=31.4% | 6 contracts
  SIGNAL KXHIGHMIA-26APR24-B80.5 | BUY NO @ 0.69 | forecast=15.9% market=69.0% edge=15.1% | 5 contracts
  SIGNAL KXHIGHTDAL-26APR24-T85 | BUY YES @ 0.04 | forecast=72.6% market=

## Place an Order
Edit the cell below then run it. Remove the `raise` line when you're ready to actually submit.

In [8]:
TICKER    = 'KXHIGHNY-26APR24-B64.5'   # market to trade
SIDE      = 'yes'                        # 'yes' or 'no'
CONTRACTS = 5                            # number of contracts
PRICE     = 35                           # limit price in cents (1–99)

raise RuntimeError("Remove this line to submit the order")

order = client.place_order(
    ticker=TICKER,
    side=SIDE,
    count=CONTRACTS,
    order_type='limit',
    yes_price=PRICE if SIDE == 'yes' else None,
    no_price=PRICE if SIDE == 'no' else None,
)
print(order)

RuntimeError: Remove this line to submit the order

## Cancel an Order

In [ ]:
ORDER_ID = 'paste-order-id-here'

raise RuntimeError("Remove this line to cancel the order")

result = client.cancel_order(ORDER_ID)
print(result)

## Orderbook — Inspect a Single Market

In [ ]:
ticker = 'KXHIGHNY-26APR24-B64.5'

ob = client.get_orderbook(ticker, depth=10)

yes_bids = ob.get('yes', [])
no_bids  = ob.get('no', [])

df_ob = pd.DataFrame({
    'YES bid (¢)': [x[0] for x in yes_bids],
    'YES size':    [x[1] for x in yes_bids],
    'NO bid (¢)':  [x[0] for x in no_bids] + [None] * max(0, len(yes_bids) - len(no_bids)),
    'NO size':     [x[1] for x in no_bids]  + [None] * max(0, len(yes_bids) - len(no_bids)),
})
df_ob

## Browse Any Market Category
Fetch open markets for any series ticker.

In [ ]:
series = 'KXRAINNYC'

result = client._get('/events', {'series_ticker': series, 'status': 'open', 'limit': 20})
events = result.get('events', [])

all_markets = []
for event in events:
    ms = client.get_markets(event_ticker=event['event_ticker'], status='open')
    all_markets.extend(ms)

pd.DataFrame([{
    'ticker':  m.get('ticker'),
    'title':   m.get('title','').replace('**',''),
    'yes_bid': float(m.get('yes_bid_dollars') or 0),
    'yes_ask': float(m.get('yes_ask_dollars') or 0),
    'volume':  float(m.get('volume_fp') or 0),
} for m in all_markets])